In [ ]:

import pandas as pd
from src.wrapper import SeededForwardSelection
import os

In [ ]:


data_dir = "data/processed/Brain"
methods = ["variance", "correlation", "chi_squared", "mutual_information",
"anova_f_test"]
n = 50

all_features = set()
for m in methods:
   path = f"{data_dir}/Brain_{m}_{n}features.csv"
   cols = pd.read_csv(path, nrows=0).columns.tolist()
   all_features.update(cols[1:])   # bỏ cột target (index 0)

union_features = sorted(all_features)
print(f"Union: {len(union_features)} unique features từ 5 methods")

In [ ]:

# Đọc từ raw/preprocessed — đủ rows, đủ cols
preprocessed = pd.read_csv("data/raw/Brain.csv")

target_col = preprocessed.columns[0]          # cột target
X_union = preprocessed[union_features]        # chỉ lấy union features
y = preprocessed[target_col]

# Ghép lại: target đầu tiên
df_union = pd.concat([y, X_union], axis=1)

save_path = f"{data_dir}/Brain_union_{n}features.csv"
df_union.to_csv(save_path, index=False)
print(f"Saved: {save_path}  shape={df_union.shape}")

In [ ]:

VOTING_CSV = f"{data_dir}/top50_features_voting_20260304_0337.csv"

selector = SeededForwardSelection(
   seed_source=VOTING_CSV,
   n_seeds=1,
   model="dt",
   scoring="accuracy",
   cv=4,                   # dùng CV vì dataset nhỏ (43 rows)
   max_features=20,
   patience=3,
   verbose=2,
)

X_in = df_union.iloc[:, 1:]
y_in = df_union.iloc[:, 0]

selector.fit(X_in, y_in)

In [ ]:
# Lấy full dataset với selected features + target
X_selected = selector.transform(X_in)          # numpy array
X_selected_df = pd.DataFrame(X_selected, columns=selector.get_feature_names_out())

df_final = pd.concat([y_in.reset_index(drop=True), X_selected_df], axis=1)

print(f"Final dataset shape: {df_final.shape}")
display(df_final.head()) #type: ignore

os.makedirs("data/processed/Brain", exist_ok=True)

save_path = "data/processed/Brain/Brain_SFS_selected.csv"
df_final.to_csv(save_path, index=False)
print(f"Saved: {save_path}")

In [ ]:

print("Selected:", selector.get_feature_names_out())
os.makedirs("result/Brain/report", exist_ok=True)
selector.save_history("result/Brain/report/sfs_history.csv")